# Exp2 evaluation: aggregated vs temporal (sequence) models

Compares models on the same 15,000 test applicants (RQ2.1), then breaks the results down by applicants' history availability and richness (RQ2.2).

**How to use**
1. Train your models on `split == 'train'` and tune on `split == 'val'` (from `clean_data/splits.csv`).
2. Predict a score for **every val and test applicant** with each model.
3. Put them in `MODELS` below and **Run All**.

If `MODELS` is left empty, the notebook runs a demo with two dummy models so you can see the output format.

In [52]:
import sys
from pathlib import Path
import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "outputs" / "exp2"] if (p / "eval_exp2.py").exists())
sys.path.insert(0, str(HERE))
sys.dont_write_bytecode = True
import eval_exp2 as ev

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Your models

In [53]:
from pathlib import Path

OUTPUT_DIR = Path("experiment2_outputs")

MODELS = {
    "M0 (Agg baseline)":      OUTPUT_DIR / "v6_pred_M0.csv",
    "M1 (Agg+Temporal)":      OUTPUT_DIR / "v6_pred_M1.csv",
    "M2 (Agg+LSTM)":          OUTPUT_DIR / "v6_pred_M2.csv",
    "M3 (Agg+Temporal+LSTM)": OUTPUT_DIR / "v6_pred_M3.csv",
    "M_app (App only)":       OUTPUT_DIR / "v6_pred_M_app.csv",
    "M_app_LSTM (App+LSTM)":  OUTPUT_DIR / "v6_pred_M_app_LSTM.csv",
}
MODELS = {k: str(v) for k, v in MODELS.items()}

N_BOOT = 1000
SAVE = True
SEGMENTS = None

In [54]:
DEMO = not MODELS
if DEMO:
    print("MODELS is empty — please fill in the MODELS dict above with your prediction files.")
else:
    print(f"Loaded {len(MODELS)} models: {', '.join(MODELS)}")
    print(f"Baseline (first model): {list(MODELS)[0]}")

# ---------------------------------------------------------------------------
# Holm correction + helpers (not in the eval_exp2.py engine)
# ---------------------------------------------------------------------------
import numpy as np

def add_holm(comps_df):
    """Add p_value and p_holm (Holm-Bonferroni adjusted) columns."""
    c = comps_df.copy()
    raw_p = np.array([row.p_delong if row.metric == "roc_auc" else row.p_bootstrap
                      for _, row in c.iterrows()])
    n = len(raw_p)
    order = np.argsort(raw_p)
    adjusted = np.empty(n)
    for rank, idx in enumerate(order):
        adjusted[rank] = raw_p[idx] * (n - rank)
    for i in range(1, n):
        adjusted[i] = max(adjusted[i], adjusted[i - 1])
    adjusted = np.minimum(adjusted, 1.0)
    p_holm = np.empty(n)
    for rank, idx in enumerate(order):
        p_holm[idx] = adjusted[rank]
    c["p_value"] = raw_p
    c["p_holm"] = p_holm
    c["verdict"] = np.where(
        p_holm < 0.05,
        np.where(c.delta > 0, "significantly better", "significantly worse"),
        "no significant difference")
    return c

def pretty_comparisons_holm(comps_df):
    """Readable comparison table with p_holm and Holm-corrected verdict."""
    if comps_df.empty:
        return comps_df
    c = comps_df.copy()
    c["difference (95% CI)"] = [f"{d:+.4f} ({lo:+.4f} to {hi:+.4f})"
                                 for d, lo, hi in zip(c.delta, c.ci_low, c.ci_high)]
    fmt_p = lambda p: f"{p:.2g}"
    c["p_value_fmt"] = c["p_value"].map(fmt_p)
    c["p_holm_fmt"] = c["p_holm"].map(fmt_p)
    base_cols = ["model", "vs_baseline", "metric", "difference (95% CI)", "p_value_fmt", "p_holm_fmt", "verdict"]
    extra = [col for col in ["grouping", "segment"] if col in c.columns]
    out = c[extra + base_cols].copy()
    out.rename(columns={"p_value_fmt": "p_value", "p_holm_fmt": "p_holm"}, inplace=True)
    return out

def run_planned_comparisons(preds, pairs, n_boot=1000):
    """Run specific pairwise comparisons (not just vs first model)."""
    rows = []
    for label, baseline in pairs:
        if label not in preds or baseline not in preds:
            print(f"  Skipping {label} vs {baseline}: model not found")
            continue
        trial = preds[label]["test"].sort_values("SK_ID_CURR")
        base = preds[baseline]["test"].sort_values("SK_ID_CURR")
        y = base.TARGET.to_numpy()
        for metric in ("pr_auc", "roc_auc"):
            c = ev.compare_models(y, trial.score.to_numpy(), base.score.to_numpy(), metric, n_boot)
            rows.append({"model": label, "vs_baseline": baseline, "metric": metric,
                         "delta": c["delta"], "ci_low": c["ci_low"], "ci_high": c["ci_high"],
                         "p_bootstrap": c["p_bootstrap"],
                         "p_delong": c["delong_roc_auc"]["p_value"] if metric == "roc_auc" else np.nan})
    return pd.DataFrame(rows)

def segment_comparisons(preds, segments, columns, n_boot=1000, min_positives=20):
    """Paired comparison vs baseline inside each segment."""
    base_label = list(preds.keys())[0]
    rows = []
    for model_label in preds:
        if model_label == base_label:
            continue
        te_model = preds[model_label]["test"].merge(segments, on="SK_ID_CURR", how="left").sort_values("SK_ID_CURR")
        te_base = preds[base_label]["test"].merge(segments, on="SK_ID_CURR", how="left").sort_values("SK_ID_CURR")
        for col in columns:
            for grp in sorted(te_model[col].dropna().unique()):
                mask = te_model[col].values == grp
                y_seg = te_base.TARGET.values[mask].astype(int)
                if y_seg.sum() < min_positives:
                    continue
                s_model = te_model.score.values[mask]
                s_base = te_base.score.values[mask]
                for metric in ("pr_auc", "roc_auc"):
                    c = ev.compare_models(y_seg, s_model, s_base, metric, n_boot)
                    rows.append({"grouping": col, "segment": grp,
                                 "model": model_label, "vs_baseline": base_label,
                                 "metric": metric, "delta": c["delta"],
                                 "ci_low": c["ci_low"], "ci_high": c["ci_high"],
                                 "p_bootstrap": c["p_bootstrap"],
                                 "p_delong": c["delong_roc_auc"]["p_value"] if metric == "roc_auc" else np.nan})
    return pd.DataFrame(rows)

print("Helper functions loaded: add_holm, pretty_comparisons_holm, run_planned_comparisons, segment_comparisons")

Loaded 6 models: M0 (Agg baseline), M1 (Agg+Temporal), M2 (Agg+LSTM), M3 (Agg+Temporal+LSTM), M_app (App only), M_app_LSTM (App+LSTM)
Baseline (first model): M0 (Agg baseline)
Helper functions loaded: add_holm, pretty_comparisons_holm, run_planned_comparisons, segment_comparisons


## 2. Results

In [55]:
table, comps, preds = ev.compare_prediction_files(list(MODELS.values()), labels=list(MODELS), n_boot=N_BOOT)

### Fusion ensembles (reconstructed from M0–M3 scores)

Three fusion strategies from the LSTM notebook (Section 15):
1. **Weighted avg (M0–M3)**: each model weighted proportional to its validation ROC-AUC
2. **Blend (M0+M2)**: grid-searched optimal mixing weight on validation
3. **Meta-learner**: regularised logistic regression on logit-transformed M0–M3 scores + segment features (has_bureau, has_internal, log_events), trained on calibration half of validation

In [56]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.special import logit as _logit
from sklearn.metrics import roc_auc_score

FUSION_MODELS = ["M0 (Agg baseline)", "M1 (Agg+Temporal)", "M2 (Agg+LSTM)", "M3 (Agg+Temporal+LSTM)"]
SEED = 42

# Align all models on the same SK_ID_CURR order
ref_val = preds[FUSION_MODELS[0]]["val"].sort_values("SK_ID_CURR").reset_index(drop=True)
ref_test = preds[FUSION_MODELS[0]]["test"].sort_values("SK_ID_CURR").reset_index(drop=True)

val_scores = {m: preds[m]["val"].sort_values("SK_ID_CURR").score.values for m in FUSION_MODELS}
test_scores = {m: preds[m]["test"].sort_values("SK_ID_CURR").score.values for m in FUSION_MODELS}
y_val = ref_val.TARGET.values
y_test = ref_test.TARGET.values

# --- Strategy 1: Validation-AUC weighted average ---
val_aucs = {m: roc_auc_score(y_val, val_scores[m]) for m in FUSION_MODELS}
total_auc = sum(val_aucs.values())
weights = {m: v / total_auc for m, v in val_aucs.items()}
print("Weighted avg weights:")
for m in FUSION_MODELS:
    print(f"  {m}: val_auc={val_aucs[m]:.4f}, weight={weights[m]:.3f}")

p_wavg_val = sum(weights[m] * val_scores[m] for m in FUSION_MODELS)
p_wavg_test = sum(weights[m] * test_scores[m] for m in FUSION_MODELS)

# --- Strategy 2: Pairwise blend M0+M2 ---
m0_key, m2_key = "M0 (Agg baseline)", "M2 (Agg+LSTM)"
best_w, best_auc = 0, 0
for w in np.arange(0, 1.01, 0.01):
    blend = w * val_scores[m2_key] + (1 - w) * val_scores[m0_key]
    auc = roc_auc_score(y_val, blend)
    if auc > best_auc:
        best_w, best_auc = w, auc
print(f"\nBlend (M0+M2): best w_M2={best_w:.2f}, val_auc={best_auc:.4f}")

p_blend_val = best_w * val_scores[m2_key] + (1 - best_w) * val_scores[m0_key]
p_blend_test = best_w * test_scores[m2_key] + (1 - best_w) * test_scores[m0_key]

# --- Strategy 3: Meta-learner ---
def safe_logit(p):
    return _logit(np.clip(p, 1e-5, 1 - 1e-5))

# Segment features
bureau_ids = set(pd.read_parquet(ev.CLEAN_DIR / "bureau.parquet", columns=["SK_ID_CURR"])["SK_ID_CURR"])
inst = pd.read_parquet(ev.CLEAN_DIR / "installments.parquet", columns=["SK_ID_CURR"])
event_count = inst.groupby("SK_ID_CURR").size()

def build_meta_X(ids, scores_dict):
    feats = {}
    for m in FUSION_MODELS:
        feats[f"logit_{m}"] = safe_logit(scores_dict[m])
    feats["has_bureau"] = np.array([uid in bureau_ids for uid in ids]).astype(float)
    has_internal = np.array([uid in set(event_count.index) for uid in ids]).astype(float)
    feats["has_internal"] = has_internal
    feats["log_events"] = np.log1p(event_count.reindex(ids, fill_value=0).values).astype(float)
    for m in ["M0 (Agg baseline)", "M2 (Agg+LSTM)", "M3 (Agg+Temporal+LSTM)"]:
        feats[f"{m}_x_events"] = feats[f"logit_{m}"] * feats["log_events"]
        feats[f"{m}_x_internal"] = feats[f"logit_{m}"] * feats["has_internal"]
    return pd.DataFrame(feats)

val_ids = ref_val.SK_ID_CURR.values
test_ids = ref_test.SK_ID_CURR.values

X_val_meta = build_meta_X(val_ids, val_scores)
X_test_meta = build_meta_X(test_ids, test_scores)

cal_pos, ev_pos = train_test_split(np.arange(len(y_val)), test_size=0.5,
                                    stratify=y_val, random_state=SEED)
meta_clf = LogisticRegression(C=0.1, max_iter=1000, random_state=SEED)
meta_clf.fit(X_val_meta.iloc[cal_pos], y_val[cal_pos])

p_meta_val = meta_clf.predict_proba(X_val_meta)[:, 1]
p_meta_test = meta_clf.predict_proba(X_test_meta)[:, 1]

print(f"\nMeta-learner: val_auc={roc_auc_score(y_val, p_meta_val):.4f}, "
      f"test_auc={roc_auc_score(y_test, p_meta_test):.4f}")

# --- Add fusion models to preds and table ---
fusion_entries = {
    "Weighted avg (M0-M3)": (p_wavg_val, p_wavg_test),
    "Blend (M0+M2)": (p_blend_val, p_blend_test),
    "Meta-learner": (p_meta_val, p_meta_test),
}

for label, (p_v, p_t) in fusion_entries.items():
    val_df = ref_val[["SK_ID_CURR", "TARGET"]].assign(score=p_v, split="val")
    test_df = ref_test[["SK_ID_CURR", "TARGET"]].assign(score=p_t, split="test")
    res = ev.evaluate(y_test, p_t, y_val=y_val, s_val=p_v, n_boot=N_BOOT)
    preds[label] = {"val": val_df, "test": test_df, "threshold": res.attrs["threshold"]}
    row = {"threshold": res.attrs["threshold"]}
    for m, r in res.iterrows():
        row[m], row[f"{m}_ci_low"], row[f"{m}_ci_high"] = r["value"], r["ci_low"], r["ci_high"]
    table.loc[label] = row

print(f"\nAdded 3 fusion models to table ({len(table)} models total)")
ev.pretty_metrics(table)

Weighted avg weights:
  M0 (Agg baseline): val_auc=0.7588, weight=0.249
  M1 (Agg+Temporal): val_auc=0.7618, weight=0.250
  M2 (Agg+LSTM): val_auc=0.7620, weight=0.250
  M3 (Agg+Temporal+LSTM): val_auc=0.7641, weight=0.251

Blend (M0+M2): best w_M2=0.86, val_auc=0.7621

Meta-learner: val_auc=0.7639, test_auc=0.7591

Added 3 fusion models to table (9 models total)


,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
M0 (Agg baseline),0.144,0.758 (0.743-0.772),0.250 (0.228-0.273),0.230 (0.214-0.248),0.438 (0.410-0.467),0.302 (0.282-0.321),0.271 (0.249-0.294),0.335 (0.312-0.362)
M1 (Agg+Temporal),0.152,0.757 (0.743-0.771),0.251 (0.229-0.274),0.237 (0.221-0.256),0.419 (0.391-0.446),0.303 (0.284-0.323),0.269 (0.247-0.293),0.333 (0.311-0.357)
M2 (Agg+LSTM),0.151,0.759 (0.744-0.772),0.248 (0.227-0.271),0.234 (0.217-0.251),0.417 (0.388-0.444),0.300 (0.280-0.319),0.263 (0.243-0.287),0.326 (0.304-0.349)
M3 (Agg+Temporal+LSTM),0.146,0.757 (0.743-0.772),0.247 (0.226-0.270),0.233 (0.217-0.251),0.429 (0.400-0.457),0.302 (0.284-0.322),0.266 (0.245-0.289),0.329 (0.307-0.354)
M_app (App only),0.166,0.741 (0.726-0.755),0.215 (0.196-0.237),0.235 (0.217-0.254),0.350 (0.324-0.376),0.281 (0.260-0.301),0.249 (0.226-0.269),0.308 (0.284-0.331)
M_app_LSTM (App+LSTM),0.166,0.749 (0.735-0.762),0.223 (0.203-0.245),0.239 (0.219-0.259),0.348 (0.322-0.375),0.284 (0.262-0.304),0.252 (0.229-0.273),0.312 (0.287-0.335)
Weighted avg (M0-M3),0.133,0.759 (0.745-0.773),0.252 (0.230-0.275),0.223 (0.209-0.239),0.478 (0.451-0.505),0.304 (0.287-0.323),0.273 (0.251-0.295),0.339 (0.313-0.360)
Blend (M0+M2),0.148,0.759 (0.744-0.772),0.249 (0.228-0.272),0.233 (0.217-0.250),0.431 (0.402-0.458),0.303 (0.283-0.321),0.265 (0.244-0.288),0.329 (0.305-0.351)
Meta-learner,0.142,0.759 (0.744-0.773),0.250 (0.229-0.274),0.231 (0.216-0.248),0.444 (0.416-0.473),0.304 (0.285-0.323),0.272 (0.251-0.297),0.337 (0.315-0.361)


### Test-set metrics
One row per model. Each cell shows **value (95% confidence interval)**. The threshold used for precision, recall and
F1 was chosen on the **validation** set (the value that maximises F1), so the test set stays untouched.

| Metric | Role | Meaning |
|---|---|---|
| `pr_auc` | **primary** | precision-recall area; the most informative with only ~8% defaulters (8% = random) |
| `roc_auc` | secondary | how well the model ranks defaulters above non-defaulters (0.5 = random, 1 = perfect); unaffected by the default rate, and the standard credit-scoring metric |
| `precision` / `recall` / `f1` | operating point | at the chosen threshold: share of flagged who default / share of defaulters flagged / balance of both |

Compare models on `pr_auc` and `roc_auc` (the significance tests below). Precision, recall and F1 depend on where each
model's threshold lands, so read them as one example operating point, not as the model comparison. The max-F1 threshold also
treats a missed defaulter and a wrongly flagged applicant as equally costly; a lender would normally set the cut-off
from the cost of each error and its review capacity.

In [57]:
ev.pretty_metrics(table)

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
M0 (Agg baseline),0.144,0.758 (0.743-0.772),0.250 (0.228-0.273),0.230 (0.214-0.248),0.438 (0.410-0.467),0.302 (0.282-0.321),0.271 (0.249-0.294),0.335 (0.312-0.362)
M1 (Agg+Temporal),0.152,0.757 (0.743-0.771),0.251 (0.229-0.274),0.237 (0.221-0.256),0.419 (0.391-0.446),0.303 (0.284-0.323),0.269 (0.247-0.293),0.333 (0.311-0.357)
M2 (Agg+LSTM),0.151,0.759 (0.744-0.772),0.248 (0.227-0.271),0.234 (0.217-0.251),0.417 (0.388-0.444),0.300 (0.280-0.319),0.263 (0.243-0.287),0.326 (0.304-0.349)
M3 (Agg+Temporal+LSTM),0.146,0.757 (0.743-0.772),0.247 (0.226-0.270),0.233 (0.217-0.251),0.429 (0.400-0.457),0.302 (0.284-0.322),0.266 (0.245-0.289),0.329 (0.307-0.354)
M_app (App only),0.166,0.741 (0.726-0.755),0.215 (0.196-0.237),0.235 (0.217-0.254),0.350 (0.324-0.376),0.281 (0.260-0.301),0.249 (0.226-0.269),0.308 (0.284-0.331)
M_app_LSTM (App+LSTM),0.166,0.749 (0.735-0.762),0.223 (0.203-0.245),0.239 (0.219-0.259),0.348 (0.322-0.375),0.284 (0.262-0.304),0.252 (0.229-0.273),0.312 (0.287-0.335)
Weighted avg (M0-M3),0.133,0.759 (0.745-0.773),0.252 (0.230-0.275),0.223 (0.209-0.239),0.478 (0.451-0.505),0.304 (0.287-0.323),0.273 (0.251-0.295),0.339 (0.313-0.360)
Blend (M0+M2),0.148,0.759 (0.744-0.772),0.249 (0.228-0.272),0.233 (0.217-0.250),0.431 (0.402-0.458),0.303 (0.283-0.321),0.265 (0.244-0.288),0.329 (0.305-0.351)
Meta-learner,0.142,0.759 (0.744-0.773),0.250 (0.229-0.274),0.231 (0.216-0.248),0.444 (0.416-0.473),0.304 (0.285-0.323),0.272 (0.251-0.297),0.337 (0.315-0.361)


<cell_type>markdown</cell_type>### Planned comparisons and p_holm

| Question | Model contrast |
|---|---|
| Do temporal features help beyond aggregates? | M1 vs M0 |
| Does the LSTM score help beyond aggregates? | M2 vs M0 |
| Does temporal + LSTM combined help? | M3 vs M0 |
| Does LSTM add value on top of temporal? | M3 vs M1 |
| Does LSTM help without aggregates? | M_app_LSTM vs M_app |
| Do historical aggregates help? | M0 vs M_app |
| Does AUC-weighted ensemble help? | Weighted avg (M0–M3) vs M0 |
| Does pairwise blending help? | Blend (M0+M2) vs M0 |
| Does the meta-learner help? | Meta-learner vs M0 |

Each contrast is tested for PR-AUC and ROC-AUC. Positive differences favour the first model. The **verdict uses the Holm-adjusted p-value** (`p_holm` < 0.05) across all 18 tests, keeping the family-wise error rate at 5%.

In [58]:
# --- Planned pairwise comparisons (like exp1) ---
PLANNED_PAIRS = [
    # Individual model contrasts
    ("M1 (Agg+Temporal)",      "M0 (Agg baseline)"),      # temporal features increment
    ("M2 (Agg+LSTM)",          "M0 (Agg baseline)"),      # LSTM increment (with aggregates)
    ("M3 (Agg+Temporal+LSTM)", "M0 (Agg baseline)"),      # combined increment
    ("M3 (Agg+Temporal+LSTM)", "M1 (Agg+Temporal)"),      # LSTM on top of temporal
    ("M_app_LSTM (App+LSTM)",  "M_app (App only)"),       # LSTM without aggregates
    ("M0 (Agg baseline)",      "M_app (App only)"),       # value of historical aggregates
    # Fusion model contrasts
    ("Weighted avg (M0-M3)",   "M0 (Agg baseline)"),      # ensemble vs baseline
    ("Blend (M0+M2)",          "M0 (Agg baseline)"),      # pairwise blend vs baseline
    ("Meta-learner",           "M0 (Agg baseline)"),      # meta-learner vs baseline
]

all_comps = run_planned_comparisons(preds, PLANNED_PAIRS, n_boot=N_BOOT)
all_comps = add_holm(all_comps)
pretty_comparisons_holm(all_comps)

,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,M1 (Agg+Temporal),M0 (Agg baseline),pr_auc,+0.0011 (-0.0057 to +0.0075),0.75,1,no significant difference
1,M1 (Agg+Temporal),M0 (Agg baseline),roc_auc,-0.0007 (-0.0037 to +0.0022),0.66,1,no significant difference
2,M2 (Agg+LSTM),M0 (Agg baseline),pr_auc,-0.0022 (-0.0078 to +0.0043),0.53,1,no significant difference
3,M2 (Agg+LSTM),M0 (Agg baseline),roc_auc,+0.0012 (-0.0015 to +0.0040),0.42,1,no significant difference
4,M3 (Agg+Temporal+LSTM),M0 (Agg baseline),pr_auc,-0.0031 (-0.0107 to +0.0041),0.41,1,no significant difference
5,M3 (Agg+Temporal+LSTM),M0 (Agg baseline),roc_auc,-0.0001 (-0.0035 to +0.0035),0.98,1,no significant difference
6,M3 (Agg+Temporal+LSTM),M1 (Agg+Temporal),pr_auc,-0.0042 (-0.0095 to +0.0014),0.13,1,no significant difference
7,M3 (Agg+Temporal+LSTM),M1 (Agg+Temporal),roc_auc,+0.0006 (-0.0021 to +0.0032),0.65,1,no significant difference
8,M_app_LSTM (App+LSTM),M_app (App only),pr_auc,+0.0079 (+0.0000 to +0.0157),0.05,0.75,no significant difference
9,M_app_LSTM (App+LSTM),M_app (App only),roc_auc,+0.0078 (+0.0035 to +0.0124),0.0011,0.017,significantly better


### Results by history segment (RQ2.2)
Same metrics for each group of applicants, each model at its own validation threshold (the same threshold for
every segment: one lending policy, not one per group).
**Compare segments on ROC-AUC** (first table): it does not depend on the default rate. PR-AUC (second table) does,
so read each segment's PR-AUC against its own `default_rate`, which is the PR-AUC of a random model.
* `HISTORY_SEGMENT`: internal (Home Credit) and/or external (bureau) history, or none.
* `INTERNAL_RICHNESS`: quartile of the amount of internal history (Q1 = least, Q4 = most).

Groups with fewer than 20 defaulters show no metrics (too few to be reliable). To use your own groups, set `SEGMENTS`
to a CSV with columns `SK_ID_CURR, segment`.

In [59]:
seg = ev.load_segments(SEGMENTS)
by_seg = ev.segment_table(preds, seg, ["CUSTOM_SEGMENT"] if SEGMENTS else ["HISTORY_SEGMENT", "INTERNAL_RICHNESS"])
ev.segment_pivot(by_seg, "roc_auc").style.format({"default_rate": "{:.2%}", "n": "{:,.0f}", "defaulters": "{:,.0f}"}, precision=3)

In [60]:
ev.segment_pivot(by_seg, "pr_auc").style.format({"default_rate": "{:.2%}", "n": "{:,.0f}", "defaulters": "{:,.0f}"}, precision=3)

### Is a model better than the baseline inside each segment? (RQ2.2)
Paired difference vs the baseline (first model) inside each segment, with 95% CI. Segments with fewer than 20
defaulters are skipped. Many tests are run here (segments x models x 2 metrics), so the verdict uses `p_holm`,
adjusted across **all** segment tests. Only claim a segment effect if its verdict is significant. This cell is the
slowest (a few minutes with `N_BOOT = 1000`).

In [61]:
seg_comps = segment_comparisons(preds, seg, ["CUSTOM_SEGMENT"] if SEGMENTS else ["HISTORY_SEGMENT", "INTERNAL_RICHNESS"],
                                n_boot=N_BOOT) if len(preds) > 1 else pd.DataFrame()
if not seg_comps.empty:
    seg_comps = add_holm(seg_comps)
pretty_comparisons_holm(seg_comps)

,grouping,segment,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,HISTORY_SEGMENT,external_only,M1 (Agg+Temporal),M0 (Agg baseline),pr_auc,+0.0076 (-0.0092 to +0.0518),0.5,1,no significant difference
1,HISTORY_SEGMENT,external_only,M1 (Agg+Temporal),M0 (Agg baseline),roc_auc,-0.0127 (-0.0355 to +0.0073),0.25,1,no significant difference
2,HISTORY_SEGMENT,internal+external,M1 (Agg+Temporal),M0 (Agg baseline),pr_auc,+0.0017 (-0.0047 to +0.0082),0.61,1,no significant difference
3,HISTORY_SEGMENT,internal+external,M1 (Agg+Temporal),M0 (Agg baseline),roc_auc,-0.0002 (-0.0035 to +0.0031),0.92,1,no significant difference
4,HISTORY_SEGMENT,internal_only,M1 (Agg+Temporal),M0 (Agg baseline),pr_auc,-0.0000 (-0.0252 to +0.0234),0.91,1,no significant difference
...,...,...,...,...,...,...,...,...,...
123,INTERNAL_RICHNESS,Q3,Meta-learner,M0 (Agg baseline),roc_auc,+0.0023 (-0.0040 to +0.0085),0.46,1,no significant difference
124,INTERNAL_RICHNESS,Q4,Meta-learner,M0 (Agg baseline),pr_auc,+0.0053 (-0.0108 to +0.0224),0.48,1,no significant difference
125,INTERNAL_RICHNESS,Q4,Meta-learner,M0 (Agg baseline),roc_auc,+0.0050 (-0.0042 to +0.0154),0.31,1,no significant difference
126,INTERNAL_RICHNESS,none,Meta-learner,M0 (Agg baseline),pr_auc,-0.0031 (-0.0191 to +0.0136),0.7,1,no significant difference


## 3. Save

In [62]:
if SAVE and not DEMO:
    table.to_csv(HERE / "exp2_metrics.csv")
    all_comps.to_csv(HERE / "exp2_comparisons.csv", index=False)
    by_seg.to_csv(HERE / "exp2_by_segment.csv", index=False)
    seg_comps.to_csv(HERE / "exp2_segment_comparisons.csv", index=False)
    print("Saved result tables in", HERE)
else:
    print("Not saved (demo mode or SAVE = False).")

Saved result tables in /Users/yichuan/Desktop/NUS/BT4012/BT4012 home credit/bt4012_group6/Exp_2
